# הדגמה ויזואלית של מסנן וינר

ניצור אות נקי ורעש לבן, נעבור לתחום התדר ונמדוד את שגיאת השחזור ואת יחס האות לרעש.

בדוגמה הסינתטית האות הנקי והרעש ידועים מראש, ולכן אפשר לחשב מהם ישירות את הספקטרומים. בנתוני אמת צריך לאמוד את הספקטרומים או להשתמש בידע מוקדם. המדדים כאן מתארים את הדוגמה הזאת בלבד.


## 1. ייבוא ספריות

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


## 2. בחירת פרמטרים

אפשר לשנות את הפרמטרים הבאים ולראות כיצד התוצאה משתנה.

In [ ]:
fs = 500
duration = 2.0

frequency_1 = 5
frequency_2 = 20

amplitude_1 = 1.0
amplitude_2 = 0.5

noise_std = 1.0

t = np.arange(0, duration, 1 / fs)


## 3. יצירת אות נקי

האות מורכב משני רכיבי סינוס בתדרים שונים.

In [ ]:
clean_signal = (
    amplitude_1 * np.sin(2 * np.pi * frequency_1 * t)
    + amplitude_2 * np.sin(2 * np.pi * frequency_2 * t)
)


## 4. יצירת רעש והוספתו לאות

In [ ]:
noise = noise_std * np.random.randn(len(t))
noisy_signal = clean_signal + noise


## 5. הצגת האות הנקי

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, clean_signal)
plt.xlabel("Time [seconds]")
plt.ylabel("Amplitude")
plt.title("Clean signal")
plt.grid(True)
plt.show()


## 6. הצגת הרעש

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, noise)
plt.xlabel("Time [seconds]")
plt.ylabel("Amplitude")
plt.title("Noise")
plt.grid(True)
plt.show()


## 7. הצגת האות הרועש

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, noisy_signal)
plt.xlabel("Time [seconds]")
plt.ylabel("Amplitude")
plt.title("Noisy measured signal")
plt.grid(True)
plt.show()


## 8. מעבר לתחום התדר

נחשב FFT של האות הנקי, הרעש והאות הרועש.

In [ ]:
N = len(t)

X = np.fft.rfft(clean_signal)
V = np.fft.rfft(noise)
Y = np.fft.rfft(noisy_signal)

frequencies = np.fft.rfftfreq(N, d=1 / fs)


## 9. ספקטרום האות הנקי

נוכל לראות פיקים סביב 5 הרץ ו־20 הרץ.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(frequencies, np.abs(X))
plt.xlim(0, 60)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude")
plt.title("Spectrum of the clean signal")
plt.grid(True)
plt.show()


## 10. ספקטרום הרעש

רעש לבן מפזר אנרגיה על פני טווח רחב של תדרים.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(frequencies, np.abs(V))
plt.xlim(0, 60)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude")
plt.title("Spectrum of the noise")
plt.grid(True)
plt.show()


## 11. ספקטרום האות הרועש

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(frequencies, np.abs(Y))
plt.xlim(0, 60)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude")
plt.title("Spectrum of the noisy signal")
plt.grid(True)
plt.show()


## 12. חישוב צפיפויות ההספק

בדוגמה לימודית זו אנו יודעים מהו האות הנקי ומהו הרעש, ולכן אפשר לחשב ישירות את הספק האות ואת הספק הרעש.

במערכת אמיתית בדרך כלל צריך לאמוד את הספק האות ואת הספק הרעש מתוך הנתונים.

In [ ]:
signal_power_spectrum = np.abs(X) ** 2
noise_power_spectrum = np.abs(V) ** 2


## 13. בניית מסנן וינר

בכל תדר המסנן בודק מה היחס בין הספק האות לבין הספק האות והרעש יחד.

כאשר האות חזק ביחס לרעש, \(H(f)\) קרוב ל־1.

כאשר הרעש חזק ביחס לאות, \(H(f)\) קרוב ל־0.

In [ ]:
epsilon = 1e-12

wiener_filter = signal_power_spectrum / (
    signal_power_spectrum + noise_power_spectrum + epsilon
)


## 14. הצגת תגובת מסנן וינר

שימי לב שהמסנן מקבל ערכים גבוהים במיוחד סביב תדרי האות.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(frequencies, wiener_filter)
plt.xlim(0, 60)
plt.ylim(-0.05, 1.05)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Wiener gain")
plt.title("Wiener filter frequency response")
plt.grid(True)
plt.show()


## 15. סינון בתחום התדר

מכפילים את ספקטרום המדידה במסנן וינר.

In [ ]:
estimated_spectrum = wiener_filter * Y
estimated_signal = np.fft.irfft(estimated_spectrum, n=N)


## 16. האות המשוחזר

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, estimated_signal)
plt.xlabel("Time [seconds]")
plt.ylabel("Amplitude")
plt.title("Signal reconstructed by the Wiener filter")
plt.grid(True)
plt.show()


## 17. השוואה בין האות הנקי, הרועש והמשוחזר

כדי לראות את ההבדל בצורה ברורה נתמקד בחצי השנייה הראשונה.

In [ ]:
samples_to_show = int(0.5 * fs)

plt.figure(figsize=(12, 4))
plt.plot(t[:samples_to_show], clean_signal[:samples_to_show], label="Clean")
plt.plot(t[:samples_to_show], noisy_signal[:samples_to_show], label="Noisy", alpha=0.7)
plt.plot(t[:samples_to_show], estimated_signal[:samples_to_show], label="Wiener estimate")
plt.xlabel("Time [seconds]")
plt.ylabel("Amplitude")
plt.title("Clean, noisy and reconstructed signals")
plt.legend()
plt.grid(True)
plt.show()


## 18. השוואת שגיאות

נחשב את ממוצע ריבוע השגיאה לפני ואחרי הסינון.

In [ ]:
mse_before = np.mean((clean_signal - noisy_signal) ** 2)
mse_after = np.mean((clean_signal - estimated_signal) ** 2)

print(f"MSE before filtering: {mse_before:.6f}")
print(f"MSE after filtering:  {mse_after:.6f}")
print(f"MSE improvement:      {mse_before / mse_after:.2f} times")


## 19. חישוב יחס אות לרעש

יחס אות לרעש מחושב כך:

\[
SNR = 10\log_{10}\left(\frac{P_{signal}}{P_{noise}}ight)
\]


In [ ]:
def calculate_snr(reference_signal, error_signal):
    signal_power = np.mean(reference_signal ** 2)
    error_power = np.mean(error_signal ** 2)
    return 10 * np.log10(signal_power / error_power)

snr_before = calculate_snr(clean_signal, noisy_signal - clean_signal)
snr_after = calculate_snr(clean_signal, estimated_signal - clean_signal)

print(f"SNR before filtering: {snr_before:.2f} dB")
print(f"SNR after filtering:  {snr_after:.2f} dB")
print(f"SNR improvement:      {snr_after - snr_before:.2f} dB")


## 20. הצגת השגיאה לפני הסינון

In [ ]:
error_before = clean_signal - noisy_signal

plt.figure(figsize=(12, 4))
plt.plot(t, error_before)
plt.xlabel("Time [seconds]")
plt.ylabel("Error")
plt.title("Estimation error before Wiener filtering")
plt.grid(True)
plt.show()


## 21. הצגת השגיאה אחרי הסינון

In [ ]:
error_after = clean_signal - estimated_signal

plt.figure(figsize=(12, 4))
plt.plot(t, error_after)
plt.xlabel("Time [seconds]")
plt.ylabel("Error")
plt.title("Estimation error after Wiener filtering")
plt.grid(True)
plt.show()


## 22. ניסוי נוסף

שני את `noise_std` לערכים כמו:

- 0.2
- 0.5
- 1.0
- 2.0

לאחר כל שינוי הריצי מחדש את המחברת.

בדקי:

1. כיצד נראה האות הרועש
2. כיצד משתנה תגובת מסנן וינר
3. כיצד משתנים ערכי MSE
4. כיצד משתנה ערך SNR


## מסקנה

מסנן וינר אינו מסנן תדרים רק לפי השאלה אם התדר גבוה או נמוך.

הוא מקבל החלטה לפי היחס הסטטיסטי בין הספק האות להספק הרעש בכל תדר.

לכן:

- בתדרים שבהם האות חזק, המסנן כמעט מעביר את המידע
- בתדרים שבהם הרעש חזק, המסנן מחליש את המידע
- התוצאה נבחרת כך שממוצע ריבוע השגיאה יהיה מינימלי
